In [ ]:
'''# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session'''

## Introduction

In this experiment, we implement a Graph Attention Network (GAT) for node classification.

Unlike GCN, which aggregates neighbor information using fixed weights, GAT introduces an attention mechanism that assigns different importance to different neighbors. This allows the model to focus on more relevant nodes during message passing.

In [2]:
!pip install torch-geometric

In [3]:
import torch
import torch_geometric

print(torch.__version__)
print(torch_geometric.__version__)

2.10.0+cu128
2.7.0


### Loading Amazon Computers Dataset

In [4]:
from torch_geometric.datasets import Amazon
dataset=Amazon(root='/kaggle/working/Amazon',name='Computers')
data=dataset[0]
print(data)


Processing...


Data(x=[13752, 767], edge_index=[2, 491722], y=[13752])


Done!


In [5]:
print(data.x.shape)
print(data.edge_index.shape)
print(data.y.shape)
print(data.y.unique())

torch.Size([13752, 767])
torch.Size([2, 491722])
torch.Size([13752])
tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])


### Loading the train test validation split

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_idx = torch.load("/kaggle/input/datasets/mrxn251/gnn-splits/train_idx.pt")
val_idx = torch.load("/kaggle/input/datasets/mrxn251/gnn-splits/val_idx.pt")
test_idx = torch.load("/kaggle/input/datasets/mrxn251/gnn-splits/test_idx.pt")
train_idx = train_idx.to(device)
val_idx = val_idx.to(device)
test_idx = test_idx.to(device)

In [8]:

data = data.to(device)

In [9]:
print(len(train_idx), len(val_idx), len(test_idx))

9626 2063 2063


### Mini-Batching with Neighbor Sampling

To efficiently train on large graphs, we use **NeighborLoader** for mini-batch training.

Instead of processing the entire graph at once, NeighborLoader samples a **subgraph** for each batch consisting of:

- **Target nodes**: Nodes for which predictions and loss are computed
- **Neighbor nodes**: Additional nodes included to provide context for message passing

### Loss Computation Strategy

Loss is computed **only on the target nodes** in each batch:

- The remaining nodes are included solely to support neighborhood aggregation

In [ ]:
'''# Install dependencies from source (this may take a few minutes)
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.10.0+cu128.html
!pip install torch-geometric

# IMPORTANT: Check if it actually installed correctly
try:
    import pyg_lib
    import torch_sparse
    print("Success: Libraries are ready!")
except ImportError:
    print("Binaries not found, trying an alternative install...")
    # If the above fails, you can try installing without the '-f' link to force a local build
    !pip install pyg-lib torch-scatter torch-sparse torch-cluster -U'''

In [10]:
import torch
import torch_sparse
import pyg_lib

print(f"PyG-Lib Version: {pyg_lib.__version__}")
print(f"Torch-Sparse Version: {torch_sparse.__version__}")

PyG-Lib Version: 0.6.0+pt210cu128
Torch-Sparse Version: 0.6.18+pt210cu128


In [13]:
from torch_geometric.loader import NeighborLoader

## GAT Model Definition

We implement a 2-layer Graph Attention Network (GAT).

- The first layer uses multi-head attention to capture diverse neighborhood information
- The second layer outputs class scores using a single attention head

Dropout and ELU activation are applied to improve generalization and learning capacity.

In [14]:
import torch.nn.functional as F
from torch_geometric.nn import GATConv
class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=8, dropout=0.6):
        super().__init__()
        
        self.dropout = dropout
        
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=dropout)

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

### Training function to train for one epoch

In [15]:
def train(model, loader, optimizer, device):
    model.train()
    total_loss = 0

    for batch in loader:
        batch = batch.to(device)

        optimizer.zero_grad()

        out = model(batch.x, batch.edge_index)

        loss = F.cross_entropy(
            out[:batch.batch_size],
            batch.y[:batch.batch_size]
        )

        loss.backward()

        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

### Evaluation function to test model performance on validation dataset

In [22]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0

    for batch in loader:
        batch = batch.to(device)

        out = model(batch.x, batch.edge_index)

        pred = out[:batch.batch_size].argmax(dim=1)

        correct += (pred == batch.y[:batch.batch_size]).sum().item()
        total += batch.batch_size

    return correct / total

In [21]:
import random
import numpy as np
def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


### Testing Function

In [31]:
from sklearn.metrics import f1_score

@torch.no_grad()
def test_with_f1(model, loader, device):
    model.eval()
    
    all_preds = []
    all_labels = []

    for batch in loader:
        batch = batch.to(device)

        out = model(batch.x, batch.edge_index)
        pred = out[:batch.batch_size].argmax(dim=1)

        all_preds.append(pred.cpu())
        all_labels.append(batch.y[:batch.batch_size].cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    acc = (all_preds == all_labels).mean()
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    micro_f1 = f1_score(all_labels, all_preds, average='micro')

    return acc, macro_f1, micro_f1

In [23]:
hidden_dims = [8, 16, 32]        # usually smaller for GAT
lrs = [0.005, 0.001]
heads_list = [2, 4, 8]           # NEW
dropouts = [0.5, 0.6]            # NEW

configs = []

for h in hidden_dims:
    for lr in lrs:
        for heads in heads_list:
            for dropout in dropouts:
                config = {
                    "hidden_dim": h,
                    "lr": lr,
                    "heads": heads,
                    "dropout": dropout,
                    "num_neighbors": [10, 10]
                }
                configs.append(config)

In [24]:
#tune no of neighbors
neighbor_options = [
    [5, 5],
    [10, 10],
    [15, 10],
    [20, 10]
]


## Experiment Pipeline

This function manages the end-to-end training and evaluation process.

- Initializes the model using the given configuration
- Constructs NeighborLoaders for mini-batch training
- Trains the model across epochs with early stopping based on validation performance
- Saves the best-performing model
- Evaluates final performance on the test set using Accuracy and F1 scores

In [25]:
def run_experiment(config, seed=0, run_test=False):
    set_seed(seed)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    epochs = 50 if not run_test else 100

    
    model = GAT(
        in_channels=data.num_features,
        hidden_channels=config["hidden_dim"],
        out_channels=dataset.num_classes,
        heads=config["heads"],
        dropout=config["dropout"]
    ).to(device)

    
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["lr"],
        weight_decay=5e-4
    )

    # loaders (same)
    train_loader = NeighborLoader(
        data,
        input_nodes=train_idx,
        num_neighbors=config["num_neighbors"],
        batch_size=128,
        shuffle=True
    )

    val_loader = NeighborLoader(
        data,
        input_nodes=val_idx,
        num_neighbors=config["num_neighbors"],
        batch_size=256,
        shuffle=False
    )

    best_val = 0
    patience = 10
    counter = 0
    save_model = run_test

    for epoch in range(1, epochs + 1):
        loss = train(model, train_loader, optimizer, device)
        val_acc = evaluate(model, val_loader, device)
        #if run_test:
         #print(f"Epoch {epoch:03d} | Train Loss: {loss:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_val:
            best_val = val_acc
            counter = 0

            if save_model:
                
                torch.save(model.state_dict(), f"/kaggle/working/best_modelGAT_{seed}.pt")
        else:
            counter += 1

        if counter >= patience:
            break

    if not run_test:
        return best_val

    # load best model
    model.load_state_dict(
        torch.load(f"/kaggle/working/best_modelGAT_{seed}.pt", map_location=device)
    )

    test_loader = NeighborLoader(
        data,
        input_nodes=test_idx,
        num_neighbors=config["num_neighbors"],
        batch_size=256,
        shuffle=False
    )

    acc, macro_f1, micro_f1 = test_with_f1(model, test_loader, device)

    return acc, macro_f1, micro_f1

### Hyperparameter Tuning (hidden dimension,lr,heads,dropout)

In [ ]:

best_val = 0
best_config = None

for config in configs:
    val_acc = run_experiment(config, seed=0, run_test=False)
    
    print(config, "→", val_acc)

    if val_acc > best_val:
        best_val = val_acc
        best_config = config

print("Best config:", best_config)

{'hidden_dim': 8, 'lr': 0.005, 'heads': 2, 'dropout': 0.5, 'num_neighbors': [10, 10]} → 0.8812409112942317
{'hidden_dim': 8, 'lr': 0.005, 'heads': 2, 'dropout': 0.6, 'num_neighbors': [10, 10]} → 0.8526417838099855
{'hidden_dim': 8, 'lr': 0.005, 'heads': 4, 'dropout': 0.5, 'num_neighbors': [10, 10]} → 0.8812409112942317
{'hidden_dim': 8, 'lr': 0.005, 'heads': 4, 'dropout': 0.6, 'num_neighbors': [10, 10]} → 0.8603974793989336
{'hidden_dim': 8, 'lr': 0.005, 'heads': 8, 'dropout': 0.5, 'num_neighbors': [10, 10]} → 0.8807561803199224
{'hidden_dim': 8, 'lr': 0.005, 'heads': 8, 'dropout': 0.6, 'num_neighbors': [10, 10]} → 0.8710615608337373
{'hidden_dim': 8, 'lr': 0.001, 'heads': 2, 'dropout': 0.5, 'num_neighbors': [10, 10]} → 0.872515753756665
{'hidden_dim': 8, 'lr': 0.001, 'heads': 2, 'dropout': 0.6, 'num_neighbors': [10, 10]} → 0.8526417838099855
{'hidden_dim': 8, 'lr': 0.001, 'heads': 4, 'dropout': 0.5, 'num_neighbors': [10, 10]} → 0.8763936015511391
{'hidden_dim': 8, 'lr': 0.001, 'heads'

### Hyperparameter tuning (Number of neighbors)

In [27]:
#Tune no of neighbors 
best_val = 0
best_neighbors = None

for neighbors in neighbor_options:
    
    config = {
        "hidden_dim": best_config["hidden_dim"],
        "lr": best_config["lr"],
        "heads":best_config["heads"],
        "dropout":best_config["dropout"],
        "num_neighbors": neighbors
    }

    val_acc = run_experiment(config, seed=0, run_test=False)

    print(f"{neighbors} → {val_acc:.4f}")

    if val_acc > best_val:
        best_val = val_acc
        best_neighbors = neighbors

print("Best neighbors:", best_neighbors)

[5, 5] → 0.8764
[10, 10] → 0.8880
[15, 10] → 0.8929
[20, 10] → 0.8977
Best neighbors: [20, 10]


In [28]:
best_config["num_neighbors"] = best_neighbors

### Final Training across 10 random seeds

In [32]:
#Final Training
acc_list = []
micro_f1_list = []
macro_f1_list=[]
for seed in range(10):
    acc, macro_f1, micro_f1 = run_experiment(best_config, seed=seed, run_test=True)
    acc_list.append(acc)
    macro_f1_list.append(macro_f1)
    micro_f1_list.append(micro_f1)

    print(f"Seed {seed}: Acc={acc:.4f}, macro_F1={macro_f1:.4f},micro_F1={micro_f1:.4f}")


Seed 0: Acc=0.8871, macro_F1=0.8885,micro_F1=0.8871
Seed 1: Acc=0.9103, macro_F1=0.9133,micro_F1=0.9103
Seed 2: Acc=0.9011, macro_F1=0.8996,micro_F1=0.9011
Seed 3: Acc=0.9103, macro_F1=0.9147,micro_F1=0.9103
Seed 4: Acc=0.9016, macro_F1=0.9082,micro_F1=0.9016
Seed 5: Acc=0.9050, macro_F1=0.9038,micro_F1=0.9050
Seed 6: Acc=0.8997, macro_F1=0.8942,micro_F1=0.8997
Seed 7: Acc=0.8938, macro_F1=0.8863,micro_F1=0.8938
Seed 8: Acc=0.9089, macro_F1=0.9070,micro_F1=0.9089
Seed 9: Acc=0.9123, macro_F1=0.9135,micro_F1=0.9123


### Final Results

In [33]:
import numpy as np

acc_mean = np.mean(acc_list)
acc_std = np.std(acc_list)

macro_f1_mean = np.mean(macro_f1_list)
macro_f1_std = np.std(macro_f1_list)

micro_f1_mean = np.mean(micro_f1_list)
micro_f1_std = np.std(micro_f1_list)

print(f"\nFinal Results:")
print(f"Accuracy: {acc_mean:.4f} ± {acc_std:.4f}")
print(f"Macro F1: {macro_f1_mean:.4f} ± {macro_f1_std:.4f}")
print(f"Micro F1: {micro_f1_mean:.4f} ± {micro_f1_std:.4f}")


Final Results:
Accuracy: 0.9030 ± 0.0077
Macro F1: 0.9029 ± 0.0099
Micro F1: 0.9030 ± 0.0077
